In [2]:
# Đọc và phân tích dữ liệu
import pandas as pd
import numpy as np

dataset = pd.read_csv('house_prices.csv')
print('Số lượng mẫu:', len(dataset))
dataset.head()

Số lượng mẫu: 100


,DienTich_m2,SoPhongNgu,KhoangCach_TT_km,TuoiNha_Nam,SoTang,GiaNha_TyVND
0,132.6,4,2.0,17,5,9.58
1,42.5,1,2.4,20,3,3.27
2,116.0,3,4.9,18,2,7.85
3,129.9,3,2.8,0,2,9.78
4,99.8,3,6.3,18,2,6.56


In [ ]:
from sklearn.model_selection import train_test_split

featurenames = ['DienTich_m2','SoPhongNgu','KhoangCach_TT_km','TuoiNha_Nam','SoTang']
X = dataset[featurenames]
y = dataset['GiaNha_TyVND']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(f"Số lượng mẫu train: {len(X_train)}, Số lượng mẫu test: {len(X_test)}")

Số lượng mẫu trend: 80, Số lượng mẫu test: 20


In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

regressor = LinearRegression()
regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_test)

print(f"Hệ số chặn: {regressor.intercept_}")
for name, coef in zip(featurenames, regressor.coef_):
    print(f"Trọng số của {name} (w): {coef}")
    
print("-" * 50)
print(f"R^2 score trên tập Train: {r2_score(y_train, regressor.predict(X_train))}")
print(f"R^2 score trên tập Test: {r2_score(y_test, y_pred)}")
print(f"RMSE trên tập Train: {np.sqrt(mean_squared_error(y_train, regressor.predict(X_train)))}")

Hệ số chặn: 1.6384472834508506
Trọng số của DienTich_m2 (w): 0.0573580469925592
Trọng số của SoPhongNgu (w): 0.3577140252167118
Trọng số của KhoangCach_TT_km (w): -0.13193932046691867
Trọng số của TuoiNha_Nam (w): -0.04868250050229865
Trọng số của SoTang (w): 0.007937783031416942
--------------------------------------------------
R^2 score trên tập Train: 0.975024021551054
R^2 score trên tập Test: 0.9711951946724923
RMSE trên tập Train: 0.3751380022567077


In [6]:
def compare(example_idx):
    x = X_test.iloc[example_idx:example_idx + 1]
    y_actual = y_test.iloc[example_idx]
    y_pred = regressor.predict(x)[0]
    diff = abs(y_actual - y_pred)
    print(
        f"Mẫu {example_idx + 1:2d}| "
        f"Diện tích: {x.iloc[0]['DienTich_m2']} | "
        f"Số phòng ngủ: {int(x.iloc[0]['SoPhongNgu'])} | "
        f"Khoảng cách đến TT: {x.iloc[0]['KhoangCach_TT_km']} | "
        f"Tuổi nhà: {int(x.iloc[0]['TuoiNha_Nam'])} | "
        f"Số tầng: {int(x.iloc[0]['SoTang'])} | "
        f"Giá nhà thực tế: {y_actual} | "
        f"Giá nhà dự đoán: {y_pred} | "
        f"Chênh lệch: {diff} | "
        f"Chênh lệch (%): {diff / y_actual * 100} |"
    )

feature_columns = [
    'DienTich_m2',
    'SoPhongNgu',
    'KhoangCach_TT_km',
    'TuoiNha_Nam',
    'SoTang'
]

df_compare = pd.DataFrame({
    'DienTich_m2': [f"{v:,.2f}" for v in X_test['DienTich_m2']],
    'SoPhongNgu': [f"{v:,.2f}" for v in X_test['SoPhongNgu']],
    'KhoangCach_TT_km': [f"{v:,.2f}" for v in X_test['KhoangCach_TT_km']],
    'TuoiNha_Nam': [f"{v:,.2f}" for v in X_test['TuoiNha_Nam']],
    'SoTang': [f"{v:,.2f}" for v in X_test['SoTang']],
    'GiaThucTe': [f"{v:,.2f}" for v in y_test],
    'GiaDuDoan': [f"{v:,.2f}" for v in regressor.predict(X_test)]
})

print("\n=== Bảng Đối chiếu Toàn bộ Tập Test (Actual vs Predicted) ===")
print(df_compare.to_string(index=False))


=== Bảng Đối chiếu Toàn bộ Tập Test (Actual vs Predicted) ===
DienTich_m2 SoPhongNgu KhoangCach_TT_km TuoiNha_Nam SoTang GiaThucTe GiaDuDoan
     114.10       3.00            16.20        9.00   4.00      6.74      6.71
      76.40       2.00            16.40        5.00   4.00      3.70      4.36
     116.00       3.00             4.90       18.00   2.00      7.85      7.86
     153.30       5.00            12.60        0.00   2.00     10.18     10.57
      45.20       1.00            16.00       12.00   2.00      1.20      1.91
     157.50       5.00            14.40        3.00   3.00     10.05     10.44
     126.60       3.00            17.10        8.00   5.00      7.66      7.37
     116.40       4.00             3.90       20.00   3.00      8.28      8.28
      86.10       3.00            19.30        7.00   3.00      5.08      4.79
      99.40       3.00            13.90        9.00   1.00      6.12      6.15
      58.10       1.00            16.80        0.00   5.00      2.48

In [7]:
import statsmodels.api as sm

# Thêm cột hệ số tự do w0 (toàn số 1: x0 = 1) vào ma trận X
X_opt = np.append(arr=np.ones((X.shape[0], 1)).astype(float), values=X, axis=1)
all_feature_names = ['Intercept (w0)'] + feature_columns

# Huấn luyện mô hình OLS và truyền xname để hiển thị rõ tên từng đặc trưng
regressor_ols = sm.OLS(endog=y, exog=X_opt).fit()
print(regressor_ols.summary(xname=all_feature_names))

                            OLS Regression Results                            
Dep. Variable:           GiaNha_TyVND   R-squared:                       0.974
Model:                            OLS   Adj. R-squared:                  0.973
Method:                 Least Squares   F-statistic:                     714.5
Date:                Sat, 03 Oct 2026   Prob (F-statistic):           4.00e-73
Time:                        08:27:12   Log-Likelihood:                -46.321
No. Observations:                 100   AIC:                             104.6
Df Residuals:                      94   BIC:                             120.3
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept (w0)       1.5455      0.182  

In [ ]:
# Các đặc trưng có [P]>t (p-value) càng thấp (tiến về 0) --> đặc trưng quan trọng
# Vì đặc trưng 'SoTang' có [P]>t == 0.6 > 0.5 --> đặc trưng này VÔ DỤNG
# Xây dựng lại mô hình sau khi loại bỏ đặc trưng 'SoTang'
feature_without_sotang = [col for col in feature_columns if col != 'SoTang']
X_without_sotang = dataset[feature_without_sotang]  

X_train_wo, X_test_wo, y_train_wo, y_test_wo = train_test_split(
    X_without_sotang,
    y,
    test_size=0.2,
    random_state=0
)

regressor_wo_sotang = LinearRegression()
regressor_wo_sotang.fit(X_train_wo, y_train_wo)
y_pred_wo_sotang = regressor_wo_sotang.predict(X_test_wo)

print('=== Mô hình sau khi bỏ đặc trưng SoTang ===')
print(f'R^2 trên tập Train: {r2_score(y_train_wo, regressor_wo_sotang.predict(X_train_wo))}')
print(f'R^2 trên tập Test: {r2_score(y_test_wo, y_pred_wo_sotang)}')
print(f'RMSE trên tập Test: {np.sqrt(mean_squared_error(y_test_wo, y_pred_wo_sotang))}')

for name, coef in zip(feature_without_sotang, regressor_wo_sotang.coef_):
    print(f'Trọng số của {name}: {coef}')

# Nếu muốn chạy OLS với tập đặc trưng mới thì làm như sau:
X_opt_wo = np.append(arr=np.ones((X_without_sotang.shape[0], 1)).astype(float), values=X_without_sotang.values, axis=1)
all_feature_names_wo = ['Intercept (w0)'] + feature_without_sotang
model_ols_wo = sm.OLS(endog=y, exog=X_opt_wo).fit()
print(model_ols_wo.summary(xname=all_feature_names_wo))

=== Mô hình sau khi bỏ đặc trưng SoTang ===
R^2 trên tập Train: 0.9750024901119485
R^2 trên tập Test: 0.9710306704330074
RMSE trên tập Test: 0.42715949509925927
Trọng số của DienTich_m2: 0.057224184946787444
Trọng số của SoPhongNgu: 0.36158028587320523
Trọng số của KhoangCach_TT_km: -0.13196487648524843
Trọng số của TuoiNha_Nam: -0.048791462852107094
                            OLS Regression Results                            
Dep. Variable:           GiaNha_TyVND   R-squared:                       0.974
Model:                            OLS   Adj. R-squared:                  0.973
Method:                 Least Squares   F-statistic:                     900.2
Date:                Sat, 03 Oct 2026   Prob (F-statistic):           1.42e-74
Time:                        08:27:19   Log-Likelihood:                -46.453
No. Observations:                 100   AIC:                             102.9
Df Residuals:                      95   BIC:                             115.9
Df Model:      

In [9]:
area_min, area_max = 120,150
bedrooms = 4
dist_min, dist_max = 0, 10
age_min, age_max = 0, 5

# Tạo các kịch bản cực biên để xác định khoảng giá
scenarios = []

for area in [area_min, area_max]:
    for dist in [dist_max, dist_min]:
        for age in [age_max, age_min]:
            row = pd.DataFrame([{
                'DienTich_m2': area,
                'SoPhongNgu': bedrooms,
                'KhoangCach_TT_km': dist,
                'TuoiNha_Nam': age
            }])
            
            pred = regressor_wo_sotang.predict(row)[0]
            scenarios.append({
                'DienTich_m2': area,
                'SoPhongNgu': bedrooms,
                'KhoangCach_TT_km': dist,
                'TuoiNha_Nam': age,
                'GiaDuDoan': pred
            })

df_scenarios = pd.DataFrame(scenarios)
print(df_scenarios.sort_values('GiaDuDoan'))

gia_min = df_scenarios['GiaDuDoan'].min()
gia_max = df_scenarios['GiaDuDoan'].max()

print("\n=== KHOẢNG GIÁ NHÀ ĐƯỢC DỰ ĐOÁN ===")
print(f"Giá thấp nhất: {gia_min:,.2f} tỷ VNĐ")
print(f"Giá cao nhất: {gia_max:,.2f} tỷ VNĐ")

# Nếu muốn lấy 1 giá tương ứng với điều kiện trung bình:
# ví dụ: diện tích 135, khoảng cách 8 km, tuổi nhà 3 năm
sample = pd.DataFrame([{
    'DienTich_m2': 135,
    'SoPhongNgu': 4,
    'KhoangCach_TT_km': 8,
    'TuoiNha_Nam': 3
}])

gia_mau = regressor_wo_sotang.predict(sample)[0]
print(f"Giá mẫu trung bình (135m2, 4 PN, 8km, 3 năm): {gia_mau:,.2f} tỷ VNĐ")


   DienTich_m2  SoPhongNgu  KhoangCach_TT_km  TuoiNha_Nam  GiaDuDoan
0          120           4                10            5   8.414236
1          120           4                10            0   8.658193
2          120           4                 0            5   9.733884
3          120           4                 0            0   9.977842
4          150           4                10            5  10.130961
5          150           4                10            0  10.374918
6          150           4                 0            5  11.450610
7          150           4                 0            0  11.694567

=== KHOẢNG GIÁ NHÀ ĐƯỢC DỰ ĐOÁN ===
Giá thấp nhất: 8.41 tỷ VNĐ
Giá cao nhất: 11.69 tỷ VNĐ
Giá mẫu trung bình (135m2, 4 PN, 8km, 3 năm): 9.63 tỷ VNĐ
